In [0]:
import pytest
from pyspark.sql import SparkSession

@pytest.fixture(scope="session")
def spark():
    spark = (SparkSession.builder
             .master("local[*]")
             .appName("car_sales_tests")
             .getOrCreate())
    return spark


In [0]:
# Testing Bronze

from pyspark.sql.types import *

def test_carsales_bronze_schema(spark):
    from transformations.bronze import car_sales_schema

    expected_fields = {
        "sale_id": StringType,
        "sale_date": TimestampType,
        "car_manufacturer": StringType,
        "model_name": StringType,
        "type": StringType,
        "fuel_type": StringType,
        "transmission_type": StringType,
        "vin": StringType,
        "price": DoubleType,
        "customer_id": IntegerType,
        "customer_name": StringType,
        "payment_mode": StringType,
        "branch_id": IntegerType,
        "country": StringType,
        "region": StringType,
        "dealer_id": IntegerType,
    }

    for field in car_sales_schema.fields:
        assert field.name in expected_fields, f"Unexpected column: {field.name}"
        assert isinstance(field.dataType, expected_fields[field.name]()), f"Column {field.name} has wrong datatype"



In [0]:
# Testing Silver

import pyspark.sql.functions as F
from pyspark.sql.types import *

def test_customers_silver_validations(spark):
    from transformations.silver import customers_silver

    input_data = [
        (1, "John Doe", "john@example.com", "1234567", "USA"),
        (2, "", "bademail", "123", "UK"),          # invalid name, email, phone
        (3, "A", "a@test.com", None, "India"),     # invalid phone
    ]

    schema = StructType([
        StructField("customer_id", IntegerType(), True),
        StructField("customer_name", StringType(), True),
        StructField("customer_email", StringType(), True),
        StructField("customer_phone", StringType(), True),
        StructField("country", StringType(), True),
    ])

    df = spark.createDataFrame(input_data, schema)

    # Simulate bronze input
    spark.sql("CREATE OR REPLACE TEMP VIEW customers_bronze AS SELECT * FROM VALUES (1, 'John Doe', 'john@example.com', '1234567', 'USA')")

    # Apply logic
    result_df = df.filter(
        (F.col("customer_id").isNotNull() & (F.col("customer_id") > 0)) &
        (F.col("customer_name").isNotNull() & (F.col("customer_name") != "")) &
        (F.col("customer_email").like("%@%")) &
        (F.length(F.col("customer_phone")) >= 7) &
        (F.col("country").isNotNull() & (F.col("country") != ""))
    )

    assert result_df.count() == 1


In [0]:
# Test Gold Data
def test_sales_count_gold(spark):
    df = spark.createDataFrame([
        ("1",),
        ("2",),
        ("3",),
    ], ["sale_id"])

    result = df.agg(F.count("sale_id").alias("sales_count")).collect()[0]["sales_count"]
    assert result == 3
